# Sedov-Taylor Blastwave (1D)

A point energy deposit into a cold, near-uniform medium drives a strong,
self-similar blast wave -- the standard test for a compressible SPH scheme's
strong-shock handling and energy conservation under extreme initial gradients.

Parameter | value
---|---
$\gamma$ | 5/3
$\rho_0$ | 1.0
$E_0$ | 1.0
`goalRadius` | 0.8

The run stops once the shock reaches `goalRadius`, a time computed from the
analytic self-similar solution (`warpSPH.caseUtils.SedovSolution`) rather than
chosen by hand -- see the parameters cell.

**Three ways to seed $E_0$** (`initialization`, see the comparison cell near
the end):
- `'singular'` deposits all of $E_0$ on the single particle nearest the
  origin -- a literal delta function, and numerically harsh at low
  resolution because the initial pressure gradient is a single-particle spike.
- `'hat'` (the default here) does the same deposit, then runs **one SPH
  interpolation pass** over that field -- summation-interpolation with the
  same finalized adaptive supports the initial condition otherwise ends on --
  which spreads the spike across one smoothing scale instead of leaving a
  single-particle delta. This used to raise `NotImplementedError`; fixing it
  was the point of moving this case into its own directory.
- `'quadrant'` spreads $E_0$ evenly over the $2^{\text{dim}}$ innermost
  particles instead, with no smoothing.

**Two reference targets** are drawn on every profile panel below: the full
self-similar solve (`SedovSolution.shockState`, dotted grey / dashed red) and
a closed-form `beta`-fit estimate of the same shock radius (dotted black /
dashed green) -- the gap between the two is itself worth watching as the
resolution changes.

Unlike `sedov_1d.py` (a thin `caseMain()` wrapper meant to just be run), this
notebook is meant to be **edited while it runs**: the initial-condition
generation below calls the same case code (`warpSPH.cases.sedov.sedovCase`/
`buildSedov`) the script does, and the step loop stays unrolled in a cell
instead of being hidden inside `warpSPH.runner.run()`. Plotting calls
`drawSedov` directly rather than going through `sedovCase.setupPlot`/
`updatePlot` -- see `01-sod/sod_1d.ipynb`'s intro for why that path does not
live-update inside a Jupyter cell in this environment.

Precision note: switching between single and double precision is controlled
in the import/configuration cell below. Because precision is set when core
modules/kernels are initialized, any precision change requires a kernel
restart before re-running the notebook.

![](outputs/06-Sedov_Taylor_Blastwave_1D.gif)

In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sedov import sedovCase, drawSedov
from warpSPH.caseUtils import buildSedov
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import exportSimulationSystem, prepExport

import os
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `sedov_1d.py`, made
# explicit and editable here. `sedovCase.defaults`/`.params` are the same
# values the CLI script starts from -- anything not overridden below just
# keeps its case default.
spec = CaseSpec(caseName=sedovCase.name, scheme=sedovCase.scheme, params=dict(sedovCase.params)) \
    .merged(**sedovCase.defaults)

spec = spec.merged(
    # --- discretisation ----------------------------------------------------
    nx=800,                    # particles across the domain in every dimension
    dim=1,
    L=2.0,

    # --- output --------------------------------------------------------
    plot=True, show=True, plotInterval=25,
    store=True, storeInterval=500,     # states mode -- one HDF5 file per stored step
    caseName='06-sedovTaylorBlastwave1D',

    # --- Sedov's own knobs ---------------------------------------------------
    params=dict(
        gamma=5 / 3, rho0=1.0, E0=1.0,
        goalRadius=0.8,             # tLimit is derived from this, not set directly
        initialization='hat',       # 'hat' | 'singular' | 'quadrant' -- see intro
        viscositySwitch='NoneSwitch',
        adaptiveSupportScheme='Owen',
        adaptiveSupportCorrections=False,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`sedovCase.buildSystem` -> `buildSedov`), not re-derived here.
# `buildSystem` also replaces `ctx.spec.tLimit` with the analytic time to
# reach `goalRadius` -- read it back from `ctx.spec`, not the local `spec`.
ctx = buildContext(sedovCase, spec)
sedovCase.configureScheme(ctx)
system = sedovCase.buildSystem(ctx)
runningState = system.initializeNewState()

print(f'goalRadius = {ctx.param("goalRadius")}, goalTime = {ctx.spec.tLimit:.4g}')
print(f'{runningState.state.positions.shape[0]} particles, dt = {float(ctx.config.dt):.4g}')

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

extraData = sedovCase.extraData(ctx, runningState)

# Direct drawSedov + plt.subplots(), not sedovCase.setupPlot -- see the intro
# cell for why. `drawSedov` clears and redraws every axis itself, so this same
# call is used for both the first frame and every update below.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    import matplotlib.pyplot as plt
    fig, axis = plt.subplots(2, 2, figsize=(9, 6), squeeze=False)
    drawSedov(ctx, runningState, (fig, axis))
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

if spec.store:
    exportSimulationSystem(ctx.exportPath, 'initialState', ctx.scheme, runningState,
                           exportAdjacency=False, stages=None, exportStagesAdjacency=False,
                           extraData=dict(extraData, frame_num=0))

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(ctx.spec.tLimit / dt)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sedovCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        drawSedov(ctx, runningState, (fig, axis))
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if spec.store and (i % spec.storeInterval == 0 or i == nSteps - 1):
        exportSimulationSystem(ctx.exportPath, f'state_{i:04d}', ctx.scheme, runningState,
                               exportAdjacency=False, stages=stepResult.stages,
                               exportStagesAdjacency=True,
                               extraData=dict(extraData, frame_num=i))

In [ ]:
if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Why `hat`: comparing the three initial conditions

All three seed the same total energy $E_0$ -- only *where* it starts matters.
`'singular'` and `'quadrant'` are exact delta-like deposits; `'hat'` is the
same `'singular'` deposit, smoothed by one SPH interpolation pass. This cell
builds all three at a small `nx` (fast, no need for the full run) and plots
their initial internal-energy profile side by side.

In [ ]:
import matplotlib.pyplot as plt

compareSpec = spec.merged(nx=101)
compareCtx = buildContext(sedovCase, compareSpec)
sedovCase.configureScheme(compareCtx)

fig2, ax2 = plt.subplots(1, 1, figsize=(6, 4))
for initialization, marker in [('singular', 'o'), ('hat', 's'), ('quadrant', '^')]:
    compareSystem = buildSedov(
        compareCtx.SimulationSystem, compareCtx.SimulationState,
        config=compareCtx.config, nx=compareSpec.nx, dim=compareSpec.dim,
        domainExtent=compareSpec.L, periodicDomain=compareSpec.periodic,
        rho0=compareSpec.param('rho0'), E0=compareSpec.param('E0'),
        initialization=initialization, gamma=compareSpec.param('gamma'),
        kernel=compareCtx.config.kernel, targetNeighbors=compareCtx.config.targetNeighbors,
        dtype=compareCtx.config.dtype, device=compareCtx.config.device)
    state0 = compareSystem.initializeNewState().state
    x = state0.positions[:, 0].detach().cpu().numpy()
    u = state0.internalEnergies.detach().cpu().numpy()
    ax2.scatter(x, u, s=10, label=initialization, marker=marker)

ax2.set_yscale('log')
ax2.set_xlim(-0.2, 0.2)
ax2.set_xlabel('x')
ax2.set_ylabel('internal energy')
ax2.legend()
ax2.set_title("Initial condition: singular spike vs. one-smoothing-scale 'hat' vs. quadrant")
fig2.tight_layout()